# [Super AI Engineer Season 6] Hackathon Week 4
## 5 Domains Hackathon: Word Segmentation

**Super AI Engineer Season 6 - Level 1 Hackathon**  
- Dataset: Word Segmentation
- จัดทำโดย: 600425-วิศิษฐ์

---
### Notebook Outline
1. Setup & Imports  
2. Data Loading & Initial Inspection  
3. Dictionary & Model Preparation  
4. Data Preprocessing  
5. Evaluation & Inference (Tokenization)
6. Prediction & Submission Generation

# 1. Setup & Imports
### 1.1 ติดตั้งไลบรารีที่จำเป็น

In [1]:
!pip install -q pythainlp deepcut datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 27.6 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 57.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.3.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.


### 1.2 นำเข้าไลบรารี

In [2]:
import numpy as np
import pandas as pd
import os
from tqdm.auto import tqdm

from pythainlp.tokenize import word_tokenize
from pythainlp.util import Trie

from datasets import load_dataset
import deepcut

2026-04-04 05:01:10.797788: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775278871.203173      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775278871.318758      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775278872.144772      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775278872.144816      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775278872.144819      55 computation_placer.cc:177] computation placer alr

# 2. Data Loading & Initial Inspection
โหลด LST20 Dataset เพื่อนำมาเป็นฐานคำศัพท์ และโหลดข้อมูลโจทย์สำหรับทำ Tokenization

In [3]:
import os

# ดาวน์โหลด LST20 Corpus จาก NECTEC โดยตรง (Official Open-D URL)
if not os.path.exists("LST20_Corpus"):
    print("Downloading AIFORTHAI LST20 Corpus from NECTEC Open-D...")
    !wget -q -O lst20.zip "https://opend-portal.nectec.or.th/dataset/d1364791-84bc-4b65-9904-79aa0aa2c5a6/resource/063e2392-1eba-4099-a732-fbaf1ba9a293/download/opend_lst20_corpus.zip"
    !unzip -q lst20.zip -d LST20_Corpus_Temp  # แตกไฟล์ชั่วคราว
    
    # ย้ายเนื้อหาให้อยู่ในชื่อโฟลเดอร์ LST20_Corpus ที่ Pipeline ข้างล่างต้องการ
    !mv LST20_Corpus_Temp/LST20_Corpus/ ./LST20_Corpus/ || mv LST20_Corpus_Temp/* ./LST20_Corpus/
    !rm -rf lst20.zip LST20_Corpus_Temp
    
    print("Extraction complete.")
else:
    print("LST20_Corpus already exists.")

Extraction complete.


In [4]:
# โหลด LST20 Corpus ฝั่ง Training, Validation, และ Test มาเตรียมเป็น Vocabulary
from datasets import load_dataset

# Path Kaggle
lst20_data_dir = "/kaggle/working/LST20_Corpus" if os.path.exists("/kaggle/working/LST20_Corpus") else "LST20_Corpus"
lst20_dataset = load_dataset('lst-nectec/lst20', trust_remote_code=True, data_dir=lst20_data_dir)
print(lst20_dataset)

# ดึงคำศัพท์ทั้งหมดจาก LST20
word_set = set()
splits = ['train', 'validation', 'test']
for split in splits:
    for words in lst20_dataset[split]['tokens']:
        for word in words:
            word_set.add(word)

print(f"Total vocabulary from LST20: {len(word_set)}")

# โหลดไฟล์โจทย์ รองรับทั้งบน Kaggle และ Local
base_path_kaggle_competition = '/kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation'

test_file = f'{base_path_kaggle_competition}/ws_test.txt'
ws_list_file = f'{base_path_kaggle_competition}/ws_list.txt' 
sample_sub_file = f'{base_path_kaggle_competition}/ws_sample_submission.csv'

with open(test_file, "r", encoding="utf-8") as f:
    test_doc = f.read()

try:
    with open(ws_list_file, "r", encoding="utf-8") as f:
        ws_list = eval(f.read())
except Exception as e:
    print(f"Warning: Could not load ws_list.txt: {e}")
    ws_list = []

sample_sub = pd.read_csv(sample_sub_file)

print(f"Test document length: {len(test_doc)} characters.")
print(f"Word Segmentation List classes: {ws_list}")
print(f"Sample Submission Data:\n", sample_sub.head())

README.md: 0.00B [00:00, ?B/s]

lst20.py: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/63310 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5620 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'fname', 'tokens', 'pos_tags', 'ner_tags', 'clause_tags'],
        num_rows: 63310
    })
    validation: Dataset({
        features: ['id', 'fname', 'tokens', 'pos_tags', 'ner_tags', 'clause_tags'],
        num_rows: 5620
    })
    test: Dataset({
        features: ['id', 'fname', 'tokens', 'pos_tags', 'ner_tags', 'clause_tags'],
        num_rows: 5250
    })
})
Total vocabulary from LST20: 54770
Test document length: 37248 characters.
Word Segmentation List classes: ['B_WORD', 'E_WORD', 'I_WORD']
Sample Submission Data:
    Id Predicted
0   1    B_WORD
1   2    I_WORD
2   3    E_WORD
3   4       NaN
4   5       NaN


# 3. Dictionary & Model Preparation
เพิ่มคลังคำศัพท์เฉพาะทาง (Custom Vocabulary) และสร้าง Trie Dictionary สำหรับช่วยเป็นพจนานุกรมให้ `newmm` / `longest`

In [5]:
# คลังคำศัพท์เสริมจากชื่อบุคคลหรือสถานที่เพื่อเพิ่มความแม่นยำ
more_vocab = [
    "ทรลักษ์", "อนุพงษ์เผ่าจินดา", "ลองวิสาโล", "เนียงพาด", "พอลซาเรือน", "จีบีซี", "วลิตโรจน", 
    "วลิต", "วิตก", "วีรชัย", "พลาศรัย", "เอี่ยมฐานนท์", "โมหำหมัด", "ซอรีสะมะแอ", "สุวรรณเกษการ", 
    "ซอรี", "กวินตรา", "โพธิจักร", "ภู่ขวัญเมือง", "ไฟติ้ง", "เจริญวัชรวิทย์", "ปลั่งพินิจ", "ลุกทุ่ง", 
    "มหาชน", "เมโท", "หยิง", "ตาราบารอฟ", "เฮย", "อุสซูริส", "อามูร์มา", "เหยหลง", "วูซูลิเจียง", 
    "วานิจ", "ปิณฑวนิช", "โกสัย", 'ดาตอร์ปิโด', 'ดารณี', 'ศิลปกุล', 'ยกุล', 'สืบวงษ์ลี', 'กระจอก', 
    'เจียโก', 'วิรายุดา', 'วงษ์เทศ', 'ขอม', 'ธม', 'เสภา', 'ออเคสตรา' ,'อรรถจักร', 'สัตยานุรักษ์'
]

lst20_words = list(word_set)
lst20_words += more_vocab
lst20_trie = Trie(set(lst20_words))

print(f"Total vocabulary size after adding custom words: {len(lst20_words)}")

Total vocabulary size after adding custom words: 54820


# 4. Data Preprocessing (Prompt Setup / Space Indexing)
หาตำแหน่งที่เป็นช่องว่าง (Space) เพื่อใช้จัดระเบียบ ID ก่อนเข้าสู่กระบวนการจัด Tag

In [6]:
# ระบุ index ตรงที่เป็นช่องว่างเพื่อจัด id_count ตอนใส่ Tag ให้ซิงค์กับโจทย์
all_space_idx = [idx for idx, c in enumerate(test_doc) if c==" "]

# ใช้ deepcut ช่วย Tokenize แบบหยาบๆ เก็บ Whitespace ออก
words = word_tokenize(test_doc, engine='deepcut', keep_whitespace=False)

# clear space 
words_clear = []
for word in words:
    words_clear += word.split(" ")

print(f"Total tokens initial generated by deepcut (cleaned spaces): {len(words_clear)}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1775278972.350152      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1775278976.505796     144 service.cc:152] XLA service 0x78e3a40024e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775278976.505832     144 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB

  72/1164 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step

I0000 00:00:1775278981.348291     144 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1164/1164 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step
Total tokens initial generated by deepcut (cleaned spaces): 8074


# 5. Evaluation & Inference
ลูปตรวจตัวอักษรเพื่อใส่ Tag `B_WORD` (คำขึ้นต้น), `I_WORD` (ตรงกลางคำ), และ `E_WORD` (คำลงท้าย)<br>
หาก OOV หรือเจอตัวเลข ให้ Fallback ไปใช้ Tokenization จาก Engine ตัวอื่น (`newmm` & `longest`)

In [7]:
def has_numbers(inputString):
    return any(char.isdigit() for char in inputString)

id_count = 0
encode_pairs = [] # โครงสร้าง: [id, word_segment_tag]
oov_words = []

for i, word in tqdm(enumerate(words_clear), total=len(words_clear)):

    # กรณี Out-Of-Vocabulary (คำไม่อยู่ใน LST20)
    if word not in lst20_words:
        oov_words.append(word)
        new_words = word_tokenize(word, engine='newmm', custom_dict=lst20_trie)
        
        # กรณีมีตัวเลข ให้ใช้ longest ช่วย
        if has_numbers(word):
            new_words = word_tokenize(word, engine='longest', custom_dict=lst20_trie)
            
        for n_word in new_words:
            for idx, c in enumerate(n_word):
                id_count += 1
                if id_count in all_space_idx:
                    id_count += 1

                if idx == 0:
                    encode_pairs.append([id_count, 'B_WORD'])
                elif (idx == len(n_word) - 1) and (len(n_word) > 1):
                    encode_pairs.append([id_count, 'E_WORD'])
                else:
                    encode_pairs.append([id_count, 'I_WORD'])

    # กรณีอยู่ใน Vocabulary ปกติ
    else:
        for idx, c in enumerate(word):
            id_count += 1
            if id_count in all_space_idx:
                id_count += 1

            if idx == 0:
                encode_pairs.append([id_count, 'B_WORD'])
            elif (idx == len(word) - 1) and (len(word) > 1):
                encode_pairs.append([id_count, 'E_WORD'])
            else:
                encode_pairs.append([id_count, 'I_WORD'])

print(f"Processed tokens into labels ({len(encode_pairs)} characters mapped).")

  0%|          | 0/8074 [00:00<?, ?it/s]

Processed tokens into labels (35182 characters mapped).


# 6. Prediction & Submission Generation
รวมตารางและแปลงกลับเป็น Format ของโจทย์ และบันทึกไฟล์ส่งประกวด

In [8]:
# นำคู่ของ Label มาใส่ Dataframe
pred = pd.DataFrame(encode_pairs)
pred.columns = ['Id', 'Predicted']

# เขียนทับบน Sample Submission เพื่อป้องกัน Id มีปัญหา
sample_sub['Predicted'] = pred['Predicted']

print("Sample prediction data:\n")
print(sample_sub.head(10))

# Export ไฟล์เซฟเป็นแบบ deepcut-newmm-longest-moreVocab 
output_filename = "5hack_ws_submission.csv"
sample_sub.to_csv(output_filename, index=False)
print(f"Saved submission {output_filename} successfully.")

Sample prediction data:

   Id Predicted
0   1    B_WORD
1   2    I_WORD
2   3    E_WORD
3   4    B_WORD
4   5    I_WORD
5   6    E_WORD
6   7    B_WORD
7   8    I_WORD
8   9    I_WORD
9  10    I_WORD
Saved submission 5hack_ws_submission.csv successfully.
